## Naive RAG over KG triples (grouped by subject)\n
\n
This notebook indexes **groups of triples** from `Bundesliga23_24_test_triples_linearized.jsonl` by **subject** (e.g. `event/<uuid>`, `match/<id>`).\n
\n
This is still naive RAG (vector search over text), but produces more useful chunks than indexing each line alone.\n

In [ ]:
# Install dependencies (run once)\n
!pip install -q chromadb requests

In [ ]:
from pathlib import Path
import json
from typing import Dict, List, Any, Tuple, Optional

import requests
import chromadb
from chromadb.config import Settings

project_root = Path('.').resolve()
jsonl_path = project_root / 'Bundesliga23_24_test_triples_linearized.jsonl'
print('Project root:', project_root)
print('JSONL exists:', jsonl_path.exists(), jsonl_path)


In [ ]:
def parse_triple_text(line_text: str) -> Tuple[str, str, str]:
    """Parse 'subject predicate object...' from a linearized triple string.

    We treat the first token as subject, second as predicate, remainder as object.
    """
    parts = line_text.strip().split(' ')
    if len(parts) < 3:
        return (parts[0] if parts else 'unknown', 'unknown', '')
    subj = parts[0]
    pred = parts[1]
    obj = ' '.join(parts[2:])
    return subj, pred, obj


def load_groups_by_subject(
    path: Path,
    max_lines: Optional[int] = 100000,
    max_subjects: Optional[int] = 20000,
) -> Dict[str, List[str]]:
    """Return mapping: subject -> list of triple lines (as strings)."""
    groups: Dict[str, List[str]] = {}
    n = 0
    with open(path, 'r', encoding='utf-8') as f:
        for raw_line in f:
            if max_lines is not None and n >= max_lines:
                break
            raw_line = raw_line.strip()
            if not raw_line:
                continue
            obj = json.loads(raw_line)
            text = obj.get('text')
            if not isinstance(text, str) or not text:
                continue
            subject, _, _ = parse_triple_text(text)
            if subject not in groups:
                if max_subjects is not None and len(groups) >= max_subjects:
                    continue
                groups[subject] = []
            groups[subject].append(text)
            n += 1
    return groups


# Quick preview
preview_groups = load_groups_by_subject(jsonl_path, max_lines=2000, max_subjects=20)
print('Subjects:', len(preview_groups))
first_subject = next(iter(preview_groups.keys()))
print('Example subject:', first_subject)
print('Lines for subject:', len(preview_groups[first_subject]))
print('\n'.join(preview_groups[first_subject][:5]))


In [ ]:
def split_group_into_chunks(subject: str, lines: List[str], max_chars: int = 3500) -> List[Dict[str, Any]]:
    """Build one or more chunk docs for a subject group."""
    header = f'subject={subject}'
    docs: List[Dict[str, Any]] = []
    current: List[str] = [header]
    current_len = len(header)
    part = 0

    for line in lines:
        add_len = len(line) + 1  # +1 for newline
        if current_len + add_len > max_chars and len(current) > 1:
            docs.append({'id': f'{subject}::part::{part}', 'text': '\n'.join(current)})
            part += 1
            current = [header]
            current_len = len(header)
        current.append(line)
        current_len += add_len

    if len(current) > 1:
        docs.append({'id': f'{subject}::part::{part}', 'text': '\n'.join(current)})
    return docs


def build_docs_grouped_by_subject(
    path: Path,
    max_lines: Optional[int] = 100000,
    max_subjects: Optional[int] = 20000,
    max_chars: int = 3500,
) -> List[Dict[str, Any]]:
    groups = load_groups_by_subject(path, max_lines=max_lines, max_subjects=max_subjects)
    docs: List[Dict[str, Any]] = []
    for subject, lines in groups.items():
        docs.extend(split_group_into_chunks(subject, lines, max_chars=max_chars))
    return docs


MAX_LINES = 80000
MAX_SUBJECTS = 30000
MAX_CHARS_PER_DOC = 3500

docs = build_docs_grouped_by_subject(
    jsonl_path,
    max_lines=MAX_LINES,
    max_subjects=MAX_SUBJECTS,
    max_chars=MAX_CHARS_PER_DOC,
)
print('Docs:', len(docs))
print('Example doc id:', docs[0]['id'])
print('Example doc (first 400 chars):\n', docs[0]['text'][:400])


In [ ]:
class OllamaEmbeddingFunction:
    """Minimal embedding function wrapper for Chroma using Ollama."""

    def __init__(self, model: str = 'nomic-embed-text', base_url: str = 'http://localhost:11434'):
        self.model = model
        self.url = f'{base_url}/api/embeddings'

    def _embed_texts(self, texts: List[str]) -> List[List[float]]:
        vectors: List[List[float]] = []
        for text in texts:
            resp = requests.post(self.url, json={'model': self.model, 'prompt': text})
            resp.raise_for_status()
            vectors.append(resp.json()['embedding'])
        return vectors

    def embed_documents(self, input: List[str]) -> List[List[float]]:
        return self._embed_texts(input)

    def embed_query(self, input):
        if isinstance(input, str):
            return self._embed_texts([input])[0]
        return self._embed_texts(input)

    def __call__(self, input: List[str]) -> List[List[float]]:
        return self.embed_documents(input)

    def name(self) -> str:
        return f'ollama-{self.model}'


In [ ]:
# Create Chroma collection (batched add)
chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))
embedding_function = OllamaEmbeddingFunction()

COLLECTION_NAME = 'kg_triples_grouped_by_subject'
RESET_COLLECTION = True  # set False to keep existing index
BATCH_SIZE = 5000

if RESET_COLLECTION:
    try:
        chroma_client.delete_collection(COLLECTION_NAME)
        print('Deleted existing collection:', COLLECTION_NAME)
    except Exception:
        pass

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_function,
)

ids_all = [d['id'] for d in docs]
docs_all = [d['text'] for d in docs]

for start in range(0, len(docs_all), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(docs_all))
    collection.add(
        ids=ids_all[start:end],
        documents=docs_all[start:end],
    )
    print(f'Indexed {end}/{len(docs_all)}')

print('Indexed total:', collection.count())


In [ ]:
def call_llama_chat(
    prompt: str,
    model: str = 'llama3.1:8b',
    base_url: str = 'http://localhost:11434',
) -> str:
    url = f'{base_url}/api/chat'
    resp = requests.post(
        url,
        json={
            'model': model,
            'messages': [{'role': 'user', 'content': prompt}],
            'stream': False,
        },
    )
    resp.raise_for_status()
    return resp.json()['message']['content']


def rag_answer(question: str, top_k: int = 10) -> str:
    results = collection.query(query_texts=[question], n_results=top_k)
    context = '\n\n'.join(results['documents'][0])

    prompt = f"""
You answer questions using ONLY the facts in the context.
If the answer is not present, say you don't know.

Context (grouped KG triples):
{context}

Question: {question}
Answer:
""".strip()

    return call_llama_chat(prompt)


In [ ]:
# Example query
question = 'For match/3895060 list the linked events and their types'
print(rag_answer(question, top_k=12))
